In [26]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

In [27]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

In [28]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .master("local[*]")
    .appName("season_sanity_events")
    .getOrCreate()
    )

In [29]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events_2022_2023 = spark.read.parquet(*season_events_parquet_file_paths)

# df_events_2022_2023 = df_events_2022_2023.withColumnsRenamed({
#     "id": "eventId",
#     "player.id": "eventPlayer.id",
#     "player.name": "eventPlayer.name",
#     "team.id": "eventTeam.id",
#     "team.name": "eventTeam.name",
# })

# # Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
# players_schema = ArrayType(
#     StructType([
#         StructField("x", FloatType(), True),
#         StructField("y", FloatType(), True),
#         StructField("player", StructType([
#             StructField("id", IntegerType(), True), 
#             StructField("name", StringType(), True)]), 
#             True),
#         StructField("visibility", StringType(), True),
#         StructField("confidence", StringType(), True),
#         StructField("jerseyNum", StringType(), True)      
#     ])
# )

# # Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
# balls_schema = ArrayType(
#     StructType([
#         StructField("x", FloatType(), True),
#         StructField("y", FloatType(), True),
#         StructField("visibility", StringType(), True)
#     ]))

# df_events_2022_2023 = df_events_2022_2023.withColumns({
#     # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
#     "homePlayers_parsed": from_json("homePlayers", players_schema),
    
#     # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
#     "awayPlayers_parsed": from_json("awayPlayers", players_schema),

#     # Cria coluna com json parseado para dicionário para dados de tracking da bola
#     "balls_parsed": from_json("balls", balls_schema)

# }).drop('homePlayers', 'awayPlayers', 'balls')

df_events_2022_2023.show(5)

+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+-------------+----------------+-----------+-------------+--------------------+--------------------+--------------------+--------------------+-----------------------+-----------------------+
|             eventId|competitionId|gameId|   season|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      details_parsed|eventSubTypeDescription|eventOutcomeDescription|
+--------------------+-------------+------+---------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+-------------+----------------+-----------+-------------+--------------------+--------------------+--------------------+--------------------

In [30]:
print('Quantidade de linhas:', df_events_2022_2023.count())

Quantidade de linhas: 945154


In [31]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_2022_2023 = df_games.filter(col('season') == '2022-2023')

df_games_2022_2023.show(5)

+------+----------+---------+----------------------+-------------+-------------+------+-----------------+-------------+---------------+--------------+-----------------+------------------+-------------+------------+
|gameId|      date|   season|teamExtraTimeStartSide|teamStartSide|    venueType|teamId|         teamName|competitionId|competitionName|opponentTeamId| opponentTeamName|       stadiumName|stadiumLength|stadiumWidth|
+------+----------+---------+----------------------+-------------+-------------+------+-----------------+-------------+---------------+--------------+-----------------+------------------+-------------+------------+
|  4620|2023-01-04|2022-2023|                 Right|         Left|    TEAM_HOME|     7|   Crystal Palace|            1| Premier League|            17|Tottenham Hotspur|     Selhurst Park|        101.0|        68.0|
|  4632|2023-01-15|2022-2023|                  Left|         Left|OPPONENT_HOME|    54|           Fulham|            1| Premier League|     

In [32]:
# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)

df_games_2022_2023 = (
    df_games_2022_2023
    .withColumns({
            # homeTeam = "team" quando o mandante é o "team" (TEAM_HOME), senão homeTeam = "opponentTeam"
            "homeTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("teamId")).otherwise(F.col("opponentTeamId")),
            "homeTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("teamName")).otherwise(F.col("opponentTeamName")),
            # homeTeamStartSide = lado que o time mandante começou:
            # - venueType == TEAM_HOME: mandante é o "team" -> usa teamStartSide direto
            # - venueType == OPPONENT_HOME: mandante é o "opponentTeam" (lado não vem direto na base) ->
            #   usa o complementar do teamStartSide (Right vira Left e vice-versa)
            "homeTeamStartSide": F.when(
                F.col("venueType") == "TEAM_HOME", F.col("teamStartSide")
                ).otherwise(
                    F.when(F.col("teamStartSide") == "Right", F.lit("Left")).otherwise(F.lit("Right"))),
            
            # opponentTeam = "opponentTeam" quando o mandante é o "team" (TEAM_HOME), senão opponentTeam = "team"
            "opponentTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("opponentTeamId")).otherwise(F.col("teamId")),
            "opponentTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("opponentTeamName")).otherwise(F.col("teamName")),
            # opponentTeamStartSide = lado que o time visitante começou:
            # - venueType == TEAM_HOME: visitante é o "opponentTeam" (lado não vem direto na base) ->
            #   usa o complementar do teamStartSide
            # - venueType == OPPONENT_HOME: visitante é o próprio "team" -> usa teamStartSide direto
            "opponentTeamStartSide": F.when(
                F.col("venueType") == "TEAM_HOME", 
                F.when(F.col("teamStartSide") == "Right", F.lit("Left")).otherwise(F.lit("Right"))
                ).otherwise(F.col("teamStartSide")),
        })
        .select(
            'gameId',
            'competitionId',
            'competitionName',
            'date',
            'season',
            'venueType',
            #'homeTeamId',
            'homeTeamName',
            #'opponentTeamId',
            'opponentTeamName',
            'homeTeamStartSide',
            'opponentTeamStartSide',
            'stadiumName',
            F.col('stadiumLength').cast("float"),
            F.col('stadiumWidth').cast("float")
        )
)

df_games_2022_2023.show(5)

+------+-------------+---------------+----------+---------+-------------+-----------------+-----------------+-----------------+---------------------+------------------+-------------+------------+
|gameId|competitionId|competitionName|      date|   season|    venueType|     homeTeamName| opponentTeamName|homeTeamStartSide|opponentTeamStartSide|       stadiumName|stadiumLength|stadiumWidth|
+------+-------------+---------------+----------+---------+-------------+-----------------+-----------------+-----------------+---------------------+------------------+-------------+------------+
|  4620|            1| Premier League|2023-01-04|2022-2023|    TEAM_HOME|   Crystal Palace|Tottenham Hotspur|             Left|                Right|     Selhurst Park|        101.0|        68.0|
|  4632|            1| Premier League|2023-01-15|2022-2023|OPPONENT_HOME| Newcastle United|           Fulham|            Right|                 Left|    St James' Park|        105.0|        68.0|
|  4656|            

In [33]:
df_events_games_2022_2023 = df_events_2022_2023.join(df_games_2022_2023, on = ["competitionId", "season", "gameId"], how='left')
df_events_games_2022_2023.show()

+-------------+---------+------+--------------------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+-------------+----------------+-----------+-------------+--------------------+--------------------+--------------------+--------------------+-----------------------+-----------------------+---------------+----------+-------------+------------+----------------+-----------------+---------------------+------------------+-------------+------------+
|competitionId|   season|gameId|             eventId|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      details_parsed|eventSubTypeDescription|eventOutcomeDescription|competitionName|      date|    venueType|homeTeamName|opponentTeamName|homeTeamStartSide|opponentTeamStartSide|       stadiumName|stadiumLength|

In [34]:
df_events_games_2022_2023 = df_events_games_2022_2023.withColumns({

    # Confere se nos dados de tracking do mandante não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_home":
    ((size(col("homePlayers_parsed")) > 0) & 
    forall(
        col("homePlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Confere se nos dados de tracking do adversário não é lista vazia e tem x,y preenchidos dos 11 jogadores (True/False convertido para binário)
    "all_tracking_away":
    ((size(col("awayPlayers_parsed")) > 0) & 
    forall(
        col("awayPlayers_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    "all_balls": 
    ((size(col("balls_parsed")) > 0) & 
    forall(
        col("balls_parsed"),
        lambda k: k.x.isNotNull() & k.y.isNotNull()
    )).cast("int"),

    # Traz a quantidade de dicionários de cada evento para saber se tem 11 jogadores do time mandante e adversário
    "len_tracking_home": size(col("homePlayers_parsed")),
    "len_tracking_away": size(col("awayPlayers_parsed"))
})

df_events_games_2022_2023.show()

+-------------+---------+------+--------------------+------+-----------------+------------+--------------------+--------------+-----------------------+--------+-------------+----------------+-----------+-------------+--------------------+--------------------+--------------------+--------------------+-----------------------+-----------------------+---------------+----------+-------------+------------+----------------+-----------------+---------------------+------------------+-------------+------------+-----------------+-----------------+---------+-----------------+-----------------+
|competitionId|   season|gameId|             eventId|period|periodDescription|   eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      details_parsed|eventSubTypeDescription|eventOutcomeDescription|competitionName|      date|    venueType|homeTeamName|opponent

In [35]:
df_acima_abaixo_11 = df_events_games_2022_2023.filter(
    ((col('len_tracking_home') != 11) & (col('len_tracking_home') > 0)) | 
    ((col('len_tracking_away') != 11) & (col('len_tracking_away') > 0)))

#df_acima_abaixo_11.cache()
df_acima_abaixo_11.show()

+-------------+---------+------+--------------------+------+-----------------+-------------+--------------------+--------------+-----------------------+--------+-------------+----------------+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------+-----------------------+---------------+----------+-------------+--------------------+--------------------+-----------------+---------------------+------------------+-------------+------------+-----------------+-----------------+---------+-----------------+-----------------+
|competitionId|   season|gameId|             eventId|period|periodDescription|    eventType|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|eventPlayerId| eventPlayerName|eventTeamId|       eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      details_parsed|eventSubTypeDescription|eventOutcomeDescription|competitionName|      date|    ven

In [36]:
df_events_games_2022_2023.select('eventTypeDescription').distinct().show(truncate=False)

+--------------------------------------+
|eventTypeDescription                  |
+--------------------------------------+
|Second half kick off                  |
|Shot                                  |
|A possession with a player on the ball|
|Challenge                             |
|First half kick off                   |
|Unknown                               |
|Clearance                             |
|Rebound                               |
|Cross                                 |
|Touch Carry                           |
|Ball Carry                            |
|Pass                                  |
+--------------------------------------+



In [37]:
# sanity dos eventos
## sanity dos periodos de cada partida
## sanity dos times com venue type
## sanity dos eventos distintos
## sanity de duplicatas de eventos removendo os identificadores dos eventos e ver eventos duplicados que aparecem (OTB é fato que repete pq reflete outros eventos)
## sanity da volumetria de outros tipos de evento 
## sanity pra verificar se os subtipos e outcome dos eventos são corretos a ele conforme a documentação
## sanity de eventos únicos na partida desnecessários (kick off, por ex)

# sanity de gameEvents
## sanity do tamanho dos estádios e se varia
## sanity do venueType
## sanity de bolas fora do campo


## outros sanity
# montar base com tracking dos 22 e bola -> analisar primeiro casos com menos ou mais de 11 dados de tracking
#registros com mais ou menos que 11 x,y do mandante e adversário -> corrigir pra concertar a tabela
#jogadores muito dispersos
#jogadores muito perto um do outro
#% de valores estimados
#% da confiança dos valores

In [38]:
# # ============================================================
# # Análise do details_parsed: uma amostra por eventTypeDescription
# # ============================================================

# df_details_sample = (
#     df_events
#     .select('eventType', 'eventTypeDescription', 'details_parsed')
#     .dropDuplicates(['eventTypeDescription'])
#     .orderBy('eventTypeDescription')
# )

# for row in df_details_sample.collect():
#     print(f"eventType={row['eventType']} | eventTypeDescription={row['eventTypeDescription']}")
#     print(row['details_parsed'])
#     print('-' * 80)

In [39]:
# dedup_subset_cols = [
#     c for c in df_events.columns
#     if c not in ('eventId', 'eventType', 'eventTypeDescription', 'eventSubTypeDescription', 'eventOutcomeDescription')
# ]

# df_dup_groups = (
#     df_events
#     .groupBy(dedup_subset_cols)
#     .agg(
#         F.count('*').alias('n_duplicates'),
#         F.sort_array(F.collect_set('eventType')).alias('eventTypes_in_group')
#     )
#     .filter(F.col('n_duplicates') > 1)
#     .select('n_duplicates', 'eventTypes_in_group')
# )

# total_dup_groups = df_dup_groups.count()
# groups_with_otb = df_dup_groups.filter(F.array_contains('eventTypes_in_group', 'OTB')).count()

# print(f'Grupos de duplicata: {total_dup_groups}')
# print(f'Grupos que contêm OTB: {groups_with_otb} ({groups_with_otb / total_dup_groups:.1%})')

# # combinações de eventType que aparecem duplicadas juntas (ex: "OTB + PASS")
# # (
# #     df_dup_groups
# #     .withColumn('combo', F.concat_ws(' + ', 'eventTypes_in_group'))
# #     .groupBy('combo')
# #     .count()
# #     .orderBy(F.desc('count'))
# #     .show(50, truncate=False)
# # )

# # ============================================================
# # Remoção das duplicatas causadas pelo eventType OTB
# # ============================================================
# # OTB é, em 99.8% dos casos, duplicata de conteúdo de outro evento (mesmo
# # tracking/métricas, eventId diferente). Removemos só o OTB quando ele é
# # duplicata de algo — mantemos OTB sozinho (sem duplicata) e as duplicatas
# # residuais que não envolvem OTB (ex: CH + PA).

# dedup_subset_cols = [
#     c for c in df_events.columns
#     if c not in ('eventId', 'eventType', 'eventTypeDescription', 'eventSubTypeDescription', 'eventOutcomeDescription')
# ]

# w_content = Window.partitionBy(*dedup_subset_cols)

# df_events_flagged = (
#     df_events
#     .withColumn('content_count', F.count('*').over(w_content))
#     .withColumn(
#         'has_non_otb_in_group',
#         F.max(F.when(F.col('eventType') != 'OTB', 1).otherwise(0)).over(w_content) == 1
#     )
# )

# # remove só o OTB que é duplicata de outro evento (mantém OTB sozinho e duplicatas sem OTB)
# df_events = (
#     df_events_flagged
#     .filter(~((F.col('eventType') == 'OTB') & (F.col('content_count') > 1)))
#     .drop('content_count', 'has_non_otb_in_group')
# )

# print('Quantidade de linhas depois de remover OTB que duplicam eventos:', df_events.count())

### Eventos que queremos (Defensivos):

- Ofensivos como CR, PA e SH queremos que o resultado dele seja uma interferência da defesa adversária
- Defensivos como CL, CH, FO queremos que o tipo seja ação defensiva

- CL: Clearance
    - Qualquer CLEARANCE_OUTCOME_TYPE (A,B,D,E,O,P,S,U)
    - obs: talvez não E e U pq são FairPlay
    - obs2: P - Player e S - Stoppage não sei oq sejam, mas vou deixar

- CR: Cross
    - CROSS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- CH: Challenge. 
    - CHALLENGE_TYPE:
    - ‘5’ - 50/50. This is a duel type where two players compete for a loose ball.
    - A - Aerial duel. As the name suggests a duel type similar to 50-50, but with the ball coming from above.
    - B - Tackle from behind. As the name suggests a tackle attempt where the carrier puts their body in between the ball and the challenger as the tackle is attempted.
    - D - Dribble. The player tries to take on a defender in an attempt to get past them.
    - G - Goalkeeper smothers ball. A duel between the goalkeeper and a line player where the ball is loose and the goalkeeper tries to capture the ball.
    - H - Shielding. Similar to tackle from behind, but on a shielding challenge the carrier actively shields a defender who does not attempt a tackle
    - K - Hand tackle by goalkeeper. Despite the name, it is a duel type similar to goalkeeper smothers, but the keeper tries to parry the ball rather than retain it.
    - L - Slide tackle. Tackle type where the challenger slides to attempt to win the ball. Note that a player could be sliding on a dribble or 50-50, to be classed as a slide tackle it needs to be first and foremost a tackle.
    - S - Shoulder to shoulder. Tackle type where the challenger tries to win the ball with physical contact initiated with the body.
    - T - Standing tackle. Tackle attempt, usually from the front or side, that does not fit the other tackle types 
    - OBS1: **Único que não entraria como AD aqui seria o 'D'.**
    - OBS2: **Não estamos considerando outcome dos eventos.**

- PA: Pass
    - PASS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- FO: Foul
    - qualquer FOUL_TYPE = A, I, M
    - não vi evento de penalti, então teria q pegar a região dentro da area e evento de falta marcado ali (FOUL_TYPE == I)

- FOUL: Additional foul
    - são faltas adicionais no mesmo lance divida em mais de um evento, mas nos dados fica tudo NULL, então n vou add. FO já tem o evento principal de falta

- SH: Shot
    - SHOT_OUTCOME_TYPE:
    - B - Block on target. (Ball was going on target, but got blocked)
    - C - Block off target. (Ball was going off target, and got blocked)
    - F - Save off target. (Ball was going off target when it got saved)
    - L - Goalline clearance. (Ball is past the goalkeeper and a defender stops it from going into the net)
    - S - Save on target. (Ball was going on target and got saved).